## 3.2 基础API的使用
LangChain 1.2.x 的长期记忆基于 store 持久化数据，相关的API有：
* put() ：负责写入
* get() ：负责读取
* search() ：负责检索

我们可以在Agent执行流程之外直接访问长期记忆

### 3.2.1 put() 写入API
① put()源码剖析

直接调用put()函数即可，函数签名如下
```python
put(
  self,
  namespace: tuple[str, ...],
  key: str,
  value: dict[str, Any],
  index: Literal[False] | list[str] | None = None,
  *,
  ttl: float | None | NotProvided = NOT_PROVIDED
) -> None
```

参数说明:

| 参数 | 类型 | 含义 |
|---|---|---|
| `namespace` | `tuple[str, ...]` | 记忆所在的命名空间，如 `("users", "alice", "memories")`。不能是空元组；每一段必须是非空字符串，且**不能含 `.`**（SQL 版 store 会用 `.` 把它拼成一个字符串存进表里）；第一段不能是 `"langgraph"`（框架保留） |
| `key` | `str` | 在该 namespace 内唯一的 ID。传入后会先 `str(key)`，所以传数字也会变成字符串 |
| `value` | `dict[str, Any]` | 记忆内容。键必须是字符串，值必须能 JSON 序列化（SQL 版要把它转成 JSON 存储） |
| `index` | `False` / `list[str]` / `None` | 只和**语义检索**有关，决定 value 里哪些字段要生成 embedding（向量）。`None`（默认）：用创建 store 时 `index` 配置里的 `fields`；`False`：这条不生成；`["text"]`：只对列出的字段生成。创建 store 时没配 `index` 的话，这个参数会被忽略 |
| `ttl` | `float` / `None` | 存活时间，单位是**分钟**，过期后会被清理。不传：用 store 的 `ttl` 配置里的 `default_ttl`（没配就是永不过期）；传 `None`：永不过期。`InMemoryStore` 不支持 TTL，传数字会抛 `NotImplementedError` |

签名里的 `*` 表示它后面的参数（`ttl`）只能用关键字传，如 `store.put(ns, key, value, ttl=10)`。

返回值：`None`。put 不会返回写好的 Item，要看结果需要再 `get()`。

put 是 **upsert**（update + insert）：namespace + key 不存在就新增，已存在就覆盖。store 没有单独的 `update()` 方法。

举例:
```python
store.put(
    ("users", "alice", "memories"), # namespace
    "pref_food", # key
    {"category": "food", "text": "Alice likes sushi"} # value
)
```

**put() 的函数体：只打包，不写数据**

put() 定义在 `BaseStore` 里。`BaseStore` 是所有 store 的抽象父类，`InMemoryStore`、`SqliteStore`、`PostgresStore` 都继承它。put() 的函数体只有三步：

```python
def put(self, namespace, key, value, index=None, *, ttl=NOT_PROVIDED):
    _validate_namespace(namespace)                       # 1. 校验 namespace
    if ttl not in (NOT_PROVIDED, None) and not self.supports_ttl:
        raise NotImplementedError(...)                   # 2. 不支持 TTL 的 store 传了 ttl 就报错
    self.batch([PutOp(namespace, str(key), value,        # 3. 打包成 PutOp，交给 batch()
                      index=index, ttl=_ensure_ttl(self.ttl_config, ttl))])
```

`PutOp` 是一个 `NamedTuple`（带字段名的元组），可以理解成一张“写入工单”，上面记着 namespace、key、value、index、ttl。`get()`、`search()`、`delete()` 的套路完全一样，分别打包成 `GetOp`、`SearchOp`、`PutOp(value=None)` 再交给 `batch()`。

`batch(ops)` 在 `BaseStore` 里是抽象方法，没有实现，由每种 store 自己实现。所以“增加和修改在内部怎么实现”，要看各个 store 的 `batch()` 怎么处理 PutOp，见 3.2.3 和 3.2.4。

![put() 的调用链](assets/09-store-put-call-chain.svg)

> 这些类虽然在 `langgraph` 包名下，但 store 只是一个独立的键值存储组件，读懂它不需要 LangGraph 的图、节点等概念。源码位置：`BaseStore`、`Item`、`PutOp` 和 `InMemoryStore` 在 `langgraph-checkpoint` 包（`langgraph/store/base`、`langgraph/store/memory`）；`SqliteStore` 在 `langgraph-checkpoint-sqlite` 包；`PostgresStore` 在 `langgraph-checkpoint-postgres` 包。

### 3.2.2 get()：读取API

② get()源码剖析

按照 namespace + key 精确查询，返回的不止是 value ，而是完整对象。即LangGraph底层将数据封装为 Item 对象 。

```python
get(
  self,
  namespace: tuple[str, ...],
  key: str,
  *,
  refresh_ttl: bool | None = None
) -> Item | None
```

参数说明:

| 参数 | 类型 | 含义 |
|---|---|---|
| `namespace` | `tuple[str, ...]` | 要读取的命名空间，必须**完整**给出（不是前缀；按前缀查要用 `search()`） |
| `key` | `str` | 要读取的 key |
| `refresh_ttl` | `bool` / `None` | 只对设置了 TTL 的条目有用：读取的同时是否让过期时间重新计时。`None`：用 store 的 `ttl` 配置里的 `refresh_on_read`（默认 `True`）。它只改过期时间 `expires_at`，**不改 `updated_at`** |

返回值：`Item | None`。找到就返回完整的 `Item` 对象，找不到返回 `None`，不会抛异常，所以下面的例子要先判断 `is not None`。

函数体同样只有一行：`return self.batch([GetOp(namespace, str(key), refresh_ttl)])[0]`。`batch()` 返回结果列表，`[0]` 取出唯一一个操作的结果。

```python
item = my_store.get(("users", "alice", "memories"), "pref_food")
if item is not None:
print(item.value)
# {'category': 'food', 'text': 'Alice likes sushi'}
```

item源码:

```python
Item(
  self,
  *,
  value: dict[str, Any],
  key: str,
  namespace: tuple[str, ...],
  created_at: datetime,
  updated_at: datetime
)
```

字段说明:

| 字段 | 类型 | 含义 |
|---|---|---|
| `value` | `dict[str, Any]` | 记忆内容，即 put 时传入的 value |
| `key` | `str` | 在 namespace 内唯一的 ID |
| `namespace` | `tuple[str, ...]` | 所在命名空间 |
| `created_at` | `datetime` | 创建时间 |
| `updated_at` | `datetime` | 最后一次写入的时间 |

* 签名里的 `*` 表示五个字段都只能用关键字传。Item 由 store 内部创建，我们一般只读取它的字段，不自己构造。
* `created_at` / `updated_at` 由 store 在写入时自动填写，put 没有参数可以指定它们。不同 store 的填写方式不同，见 3.2.4 末尾的小结。
* 构造函数里，如果传入的时间是字符串，会用 `datetime.fromisoformat()` 转成 `datetime`。SQLite 读出来的时间就是字符串，靠这一步才变成 `datetime`。
* `print(item)` 时 namespace 显示成列表 `['users', 'China']`，是因为 `__repr__` 调用了 `Item.dict()`，后者为了方便转 JSON 把元组转成了列表。`item.namespace` 本身仍是元组。
* `search()` 返回的是 `SearchItem`，它是 Item 的子类，多一个 `score` 字段（语义检索的相似度分数，不用 query 检索时为 `None`）。

### 3.2.3 基于InMemoryStore

1. 添加

In [1]:
from langgraph.store.memory import InMemoryStore


store = InMemoryStore()

namespace = ("users","China")
user_id = "user-1"
user_name = "小明"

store.put(namespace,user_id,{"name":user_name})

item = store.get(namespace,user_id)
print(item)


Item(namespace=['users', 'China'], key='user-1', value={'name': '小明'}, created_at='2026-10-08T08:43:20.819764+00:00', updated_at='2026-10-08T08:43:20.819767+00:00')


In [2]:
user_name = "小红"

store.put(namespace,user_id,{"name":user_name})

item = store.get(namespace,user_id)
print(item)

Item(namespace=['users', 'China'], key='user-1', value={'name': '小红'}, created_at='2026-10-08T08:46:39.557840+00:00', updated_at='2026-10-08T08:46:39.557842+00:00')


注意：对于当前版本，InMemoryStore每次put都会创建一个新的Item对象，无论namespace和key是否相同，**所以 created_at 和 updated_at 始终是一致的。**

**InMemoryStore 内部如何实现增加和修改**

InMemoryStore 的全部数据就是 `__init__` 里创建的一个两层字典：

```python
self._data: dict[tuple[str, ...], dict[str, Item]] = defaultdict(dict)
#                namespace             key  Item
```

`batch()` 收到 PutOp 后分两步：

1. `_prepare_ops()`：把 PutOp 按 `(namespace, key)` 放进一个字典。同一批里对同一个 key 写了多次的话，后面的覆盖前面的，只执行最后一次。
2. `_apply_put_ops()`：真正写入 `_data`：

```python
def _apply_put_ops(self, put_ops):
    for (namespace, key), op in put_ops.items():
        if op.value is None:                          # delete() 走这里
            self._data[namespace].pop(key, None)
            self._vectors[namespace].pop(key, None)   # 同时删掉语义检索用的向量
        else:                                         # 新增和修改都走这里
            self._data[namespace][key] = Item(
                value=op.value,
                key=key,
                namespace=namespace,
                created_at=datetime.now(timezone.utc),
                updated_at=datetime.now(timezone.utc),
            )
```

从这段代码可以看出：

* **新增和修改是同一行代码** `self._data[namespace][key] = Item(...)`：key 不存在时是往字典里加一项，存在时是把旧 Item 换成新 Item。代码不区分这两种情况，也不读旧 Item，所以旧的 `created_at` 就丢了。
* `created_at` 和 `updated_at` 是**两次独立的** `datetime.now()` 调用，严格说会差几微秒（上面输出里的 `...819764` 和 `...819767`），只是“几乎一致”。时间是 UTC，比北京时间晚 8 小时。
* `_data` 是 `defaultdict(dict)`：第一次往某个 namespace 写入时会自动建好内层字典，所以不需要事先“创建 namespace”。
* `get()` 返回的是字典里那个 Item 对象本身（`self._data[namespace].get(key)`），不是副本。直接改 `item.value` 会改到 store 里的数据，但这绕过了 put，`updated_at` 不会变。

下面直接查看 `_data` 来验证：

In [ ]:
demo = InMemoryStore()
ns = ("users", "China")

demo.put(ns, "user-1", {"name": "小明"})
old = demo.get(ns, "user-1")
demo.put(ns, "user-1", {"name": "小红"})   # 同一个 key 再写一次
new = demo.get(ns, "user-1")

print("底层字典的结构:", {namespace: list(items) for namespace, items in demo._data.items()})
print("旧 Item:", old.value, old.created_at)
print("新 Item:", new.value, new.created_at)
print("是同一个对象吗:", old is new)

# get 返回的就是字典里那个 Item 对象，直接改它的 value 会改到 store 里
new.value["age"] = 18
again = demo.get(ns, "user-1")
print("直接改 item.value 后再 get:", again.value, "| updated_at 变了吗:", again.updated_at != new.updated_at)

底层字典的结构: {('users', 'China'): ['user-1']}
旧 Item: {'name': '小明'} 2026-10-08 09:18:22.760682+00:00
新 Item: {'name': '小红'} 2026-10-08 09:18:22.760703+00:00
是同一个对象吗: False
直接改 item.value 后再 get: {'name': '小红', 'age': 18} | updated_at 变了吗: False


### 3.2.4 基于SQliteStore 和 基于PostgresStore
PostgresStore不同于InMemoryStore，更改数据的逻辑是update而非覆盖，因此 created_at 固定为Item 创建时间，而 updated_at 为更新时间，二者可能不同，符合直觉。


In [4]:
from langgraph.store.sqlite import SqliteStore
from pathlib import Path

db_path = Path(
    "/Users/skk/Developer/lang-chain-learn/Chapter09_memory/data/memory.sqlite"
)

with SqliteStore.from_conn_string(db_path) as store:
    store.setup()

    namespace = ("users","China")
    user_id = "user-1"
    user_name = "小明"

    store.put(namespace,user_id,{"name":user_name})

    item = store.get(namespace,user_id)
    print(item)

Item(namespace=['users', 'China'], key='user-1', value={'name': '小明'}, created_at='2026-10-08T08:51:06', updated_at='2026-10-08T08:51:06')


In [5]:
with SqliteStore.from_conn_string(db_path) as store:
    store.setup()

    namespace = ("users","China")
    user_id = "user-1"
    user_name = "小红"

    store.put(namespace,user_id,{"name":user_name})

    item = store.get(namespace,user_id)
    print(item)

Item(namespace=['users', 'China'], key='user-1', value={'name': '小红'}, created_at='2026-10-08T08:53:03', updated_at='2026-10-08T08:53:03')


**为什么 SqliteStore 修改后 created_at 也变了？**

上面两次运行中，第二次 put 后 `created_at` 从 `08:51:06` 变成了 `08:53:03`，和 InMemoryStore 一样被重置了。原因在 SQL 版 store 内部写数据的方式。

SqliteStore 和 PostgresStore 都把数据存在一张名为 `store` 的表里。表由 `setup()` 创建；`setup()` 还会建一张 `store_migrations` 表，记录建表脚本已经执行到哪一步，所以重复调用不会重复建表。`store` 表的结构：

| 列 | 内容 |
|---|---|
| `prefix` | namespace 用 `.` 拼成的字符串，如 `"users.China"`。这就是 namespace 的每一段不能含 `.` 的原因 |
| `key` | key |
| `value` | value 转成的 JSON |
| `created_at` / `updated_at` | 写入时由数据库函数 `CURRENT_TIMESTAMP` 填写 |
| `expires_at` / `ttl_minutes` | TTL 用 |

主键是 `(prefix, key)`，即 namespace + key 唯一确定一行。

`batch()` 先开启一个事务，再由 `_prepare_batch_PUT_queries()` 把 PutOp 翻译成 SQL：同一批里同一个 key 只保留最后一次；value 为 `None` 的变成 `DELETE`，其余的拼成一条插入语句。两种数据库的区别就在这条插入语句上。

SqliteStore：
```sql
INSERT OR REPLACE INTO store (prefix, key, value, created_at, updated_at, expires_at, ttl_minutes)
VALUES (?, ?, ?, CURRENT_TIMESTAMP, CURRENT_TIMESTAMP, ?, ?)
```

PostgresStore（不带 ttl 时）：
```sql
INSERT INTO store (prefix, key, value, created_at, updated_at, expires_at, ttl_minutes)
VALUES (%s, %s, %s, CURRENT_TIMESTAMP, CURRENT_TIMESTAMP, NULL, %s)
ON CONFLICT (prefix, key) DO UPDATE
SET value = EXCLUDED.value,
    updated_at = CURRENT_TIMESTAMP,
    expires_at = EXCLUDED.expires_at,
    ttl_minutes = EXCLUDED.ttl_minutes
```

* **`INSERT OR REPLACE`**（SQLite）：主键冲突时，**先删除旧行，再插入新行**。新行的 `created_at` 来自 VALUES 里的 `CURRENT_TIMESTAMP`，所以被重置，效果和 InMemoryStore 一样。
* **`ON CONFLICT ... DO UPDATE`**（Postgres）：主键冲突时，**不插入新行，而是在原行上执行 SET 子句**。`EXCLUDED` 指“本来要插入的那一行”。SET 里没有 `created_at`，所以它保留首次插入的时间；`updated_at` 改成当前时间。这才是本节开头说的“update 而非覆盖”。

![对同一个 key 写两次](assets/10-store-overwrite-timestamps.svg)

下面用内存版 SQLite 数据库验证 SqliteStore 是“删旧行、插新行”。`rowid` 是 SQLite 自动给每一行分配的行号，行号变了，说明已经是新的一行：

In [ ]:
import time

with SqliteStore.from_conn_string(":memory:") as demo:   # ":memory:" 表示内存数据库，不写文件
    demo.setup()
    ns = ("users", "China")
    sql = "SELECT rowid, prefix, key, CAST(value AS TEXT), created_at, updated_at FROM store"

    demo.put(ns, "user-1", {"name": "小明"})
    print("第一次 put 后:", demo.conn.execute(sql).fetchall())

    time.sleep(1.1)   # CURRENT_TIMESTAMP 只精确到秒，等 1 秒才看得出差别
    demo.put(ns, "user-1", {"name": "小红"})
    print("第二次 put 后:", demo.conn.execute(sql).fetchall())

第一次 put 后: [(1, 'users.China', 'user-1', '{"name":"小明"}', '2026-10-08 09:18:22', '2026-10-08 09:18:22')]


第二次 put 后: [(2, 'users.China', 'user-1', '{"name":"小红"}', '2026-10-08 09:18:23', '2026-10-08 09:18:23')]


Postgres 服务器已经关掉了，没法直接运行 PostgresStore。SQLite 也支持 `ON CONFLICT DO UPDATE` 语法（只是 SqliteStore 没有用它），所以下面在一张简化的表上对比这两种写法：

In [ ]:
import sqlite3

conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE t (key TEXT PRIMARY KEY, value TEXT, created_at TEXT, updated_at TEXT)")

# 写法一：SqliteStore 用的 INSERT OR REPLACE
conn.execute("INSERT INTO t VALUES ('user-1', '小明', '10:00', '10:00')")
conn.execute("INSERT OR REPLACE INTO t VALUES ('user-1', '小红', '10:05', '10:05')")
print("INSERT OR REPLACE    :", conn.execute("SELECT * FROM t").fetchall())

# 写法二：PostgresStore 用的 ON CONFLICT DO UPDATE
conn.execute("DELETE FROM t")
conn.execute("INSERT INTO t VALUES ('user-1', '小明', '10:00', '10:00')")
conn.execute("""
    INSERT INTO t VALUES ('user-1', '小红', '10:05', '10:05')
    ON CONFLICT (key) DO UPDATE SET value = EXCLUDED.value, updated_at = EXCLUDED.updated_at
""")
print("ON CONFLICT DO UPDATE:", conn.execute("SELECT * FROM t").fetchall())

INSERT OR REPLACE    : [('user-1', '小红', '10:05', '10:05')]
ON CONFLICT DO UPDATE: [('user-1', '小红', '10:00', '10:05')]


#### 小结：不同 API 对 created_at / updated_at 的处理

| 操作 | InMemoryStore | SqliteStore | PostgresStore |
|---|---|---|---|
| `put` 新 key | 两个时间都取当前时间（两次 `datetime.now()`，相差几微秒） | 两个都是 `CURRENT_TIMESTAMP`，完全相同 | 两个都是 `CURRENT_TIMESTAMP`，完全相同 |
| `put` 已有 key | 两个都重置（换成新 Item） | 两个都重置（删旧行、插新行） | **`created_at` 保留**，`updated_at` 更新 |
| `put` 的 value 和原来一样 | 照样重置，不比较内容 | 照样重置 | 照样更新 `updated_at` |
| `get` / `search` | 不变 | 不变（`refresh_ttl` 只改 `expires_at`） | 不变（同左） |
| `delete` | 从字典里删掉 | 执行 `DELETE`，整行删除 | 执行 `DELETE`，整行删除 |
| 直接修改 `get` 返回的 `item.value` | **会改到 store 里的数据**，但时间不变 | 不影响数据库 | 不影响数据库 |
| `search` 不带 query 时的顺序 | 字典插入顺序（覆盖写不改变位置） | `ORDER BY updated_at DESC` | `ORDER BY updated_at DESC` |
| 时间格式 | UTC，带时区，精确到微秒 | UTC，**不带时区**，精确到秒 | 带时区，精确到微秒 |

三种 store 里，只有 PostgresStore 的 `created_at` 是真正的“首次创建时间”。